# Figures 5 and 7: Storm Xaver and Hurricane Fiona maps

ERA5 fields, wind stress and surge for each experiment, and the differences between experiments, at the time of peak surge. Figure 5 is storm Xaver (North Sea, December 2013); Figure 7 is hurricane Fiona (Atlantic Canada, September 2022).

From Özkan et al. (2025), *Sensitivity of global storm surge modelling to sea surface drag*, Ocean Dynamics, https://doi.org/10.1007/s10236-025-01713-3

The processed input files are not included in this repository; this notebook shows how the figure was made. Experiment names: in the code, `cc` = FIX, `cv` = NST, `ntr`/`uv10n` = CAD and `rho`/`uv10rho` = IFS (see the main README).

In [ ]:
# Folder with the processed input files (not included in this repository)
DATA_DIR = "path/to/paper1_data/"

In [ ]:
import os
import xarray as xr
import numpy as np
import proplot as pplt
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

In [ ]:
data_base_dir  = DATA_DIR + "data/4-Processed_for_Visualization/1_storm_cases/"

# Define storm information
storms = {
    "Xaver": {
        "nc_read": "201312040000_201312080000",
        "landfall": "2013-12-06 05:00:00",
        "analysis_time": ["2013-12-04 00:00:00", "2013-12-08 00:00:00"],
        "filter_lon_lat_limits": [-5, 20, 48, 65],
        "g3_lon_lat_limits": [2, 10, 51.30, 56],
        "plot_lon_lat_limits": [-0.01, 9.01, 51.15, 58.20], # lonmin, lonmax, latmin, latmax
        "plot_lon_0": 5, "plot_lat_0": 56,
        "lonlocator": [0, 2, 4, 6, 8,],
        "latlocator": [52, 54, 56, 58, 60],
        "levels_wind": np.arange(0, 33, 3),
        "levels_charnock": np.arange(0, 110, 10), # If extend="max" given max limits are not included on the colorbar (included the one before max)
        "levels_rho": np.arange(1.120, 1.290, 0.010),
        "levels_tau": np.arange(0, 3.3, 0.3),
        "levels_surge": np.arange(0, 3.4, 0.1),
        "ticks_rho" : np.arange(1.120, 1.290, 0.020),
        "ticks_tau": np.arange(0, 3.3, 0.3),
        "ticks_surge": np.arange(0, 3.4, 0.3),
        "wind_interval": 3, "wind_scale": 200, "wind_width": 0.0065,
        "tau_interval": 3, "tau_scale": 20, "tau_width": 0.0065,
        "tau_diff1": np.arange(-0.6, 0.61, 0.1), "ticks_tau_diff1": np.arange(-0.6, 0.61, 0.2),
        "tau_diff2": np.arange(-0.2, 0.21, 0.05), "ticks_tau_diff2": np.arange(-0.2, 0.21, 0.1),
        "surge_diff": np.arange(-0.2, 0.210, 0.020), "ticks_surge_diff": np.arange(-0.2, 0.210, 0.040),
    },
    "Fiona": {
        "nc_read": "202209220000_202209260000",
        "landfall": "2022-09-24 10:00:00",
        "analysis_time": ["2022-09-22 00:00:00", "2022-09-26 00:00:00"],
        "filter_lon_lat_limits": [-74, -50, 36, 55],
        "g3_lon_lat_limits": [-68, -54, 40, 52],
        "plot_lon_lat_limits": [-67.05, -53.80, 41.60, 51.10], # lonmin, lonmax, latmin, latmax
        "plot_lon_0": -60, "plot_lat_0": 46,
        "lonlocator": [-66, -64, -62, -60, -58, -56, -54],
        "latlocator": [42, 44, 46, 48, 50, 52],
        "levels_wind": np.arange(0, 33, 3),
        "levels_charnock": np.arange(0, 110, 10),
        "levels_rho": np.arange(1.120, 1.290, 0.010),
        "levels_tau": np.arange(0, 5.5, 0.3),
        "levels_surge": np.arange(-0.4, 2.1, 0.1),
        "ticks_rho" : np.arange(1.120, 1.290, 0.020),
        "ticks_tau": np.arange(0, 5.5, 0.6),
        "ticks_surge": [-0.4, -0.2, 0, 0.2, 0.4, 0.6, 0.8, 1, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2],
        "wind_interval": 4, "wind_scale": 220, "wind_width": 0.0065,
        "tau_interval": 3, "tau_scale": 70, "tau_width": 0.0045,
        "tau_diff1": np.arange(-1.6, 0.5, 0.1), "ticks_tau_diff1": np.arange(-1.6, 0.5, 0.1),
        "tau_diff2": np.arange(-0.4, 0.5, 0.1), "ticks_tau_diff2": np.arange(-0.4, 0.5, 0.1),
        "surge_diff": np.arange(-0.3, 0.35, 0.05), "ticks_surge_diff": np.arange(-0.3, 0.35, 0.1),
    }
}

In [ ]:
# Function to load data for a storm
def load_data(storm_name, nc_read):
    data_dir = os.path.join(data_base_dir, storm_name)
    datasets = {}
    datasets["msl"] = xr.open_dataset(os.path.join(data_dir, f"ERA5_MSL_{nc_read}.nc")) / 100  # Convert to hPa
    datasets["uv10_wind"] = xr.open_dataset(os.path.join(data_dir, f"ERA5_UV10_{nc_read}.nc"))
    datasets["charnock"] = xr.open_dataset(os.path.join(data_dir, f"ERA5_C_{nc_read}.nc")) * 1000  # Convert to 1e-3
    datasets["uv10n_wind"] = xr.open_dataset(os.path.join(data_dir, f"ERA5_UV10N_{nc_read}.nc"))
    datasets["uv10rho_wind"] = xr.open_dataset(os.path.join(data_dir, f"ERA5_UV10RHO_{nc_read}.nc"))
    datasets["rho"] = xr.open_dataset(os.path.join(data_dir, f"ERA5_RHO_{nc_read}.nc"))
    datasets["cc_cd_tau"] = xr.open_dataset(os.path.join(data_dir, f"ERA5_CC_CD_TAU_{nc_read}.nc"))
    datasets["cc_taux_tauy"] = xr.open_dataset(os.path.join(data_dir, f"ERA5_CC_TAUX_TAUY_{nc_read}.nc"))
    datasets["cv_cd_tau"] = xr.open_dataset(os.path.join(data_dir, f"ERA5_CV_CD_TAU_{nc_read}.nc"))
    datasets["cv_taux_tauy"] = xr.open_dataset(os.path.join(data_dir, f"ERA5_CV_TAUX_TAUY_{nc_read}.nc"))
    datasets["uv10n_cd_tau"] = xr.open_dataset(os.path.join(data_dir, f"ERA5_UV10N_CD_TAU_{nc_read}.nc"))
    datasets["uv10n_taux_tauy"] = xr.open_dataset(os.path.join(data_dir, f"ERA5_UV10N_TAUX_TAUY_{nc_read}.nc"))
    datasets["uv10rho_cd_tau"] = xr.open_dataset(os.path.join(data_dir, f"ERA5_UV10RHO_CD_TAU_{nc_read}.nc"))
    datasets["uv10rho_taux_tauy"] = xr.open_dataset(os.path.join(data_dir, f"ERA5_UV10RHO_TAUX_TAUY_{nc_read}.nc"))
    datasets["surge_his"] = xr.open_dataset(os.path.join(data_dir, f"{storm_name}_surge_his.nc"))
    datasets["cc_map"] = xr.open_dataset(os.path.join(data_dir, f"{storm_name}_CC_surge_map.nc"))
    datasets["cv_map"] = xr.open_dataset(os.path.join(data_dir, f"{storm_name}_CV_surge_map.nc"))
    datasets["uv10n_map"] = xr.open_dataset(os.path.join(data_dir, f"{storm_name}_UV10N_surge_map.nc"))
    datasets["uv10rho_map"] = xr.open_dataset(os.path.join(data_dir, f"{storm_name}_UV10RHO_surge_map.nc"))
    print(f"Data loaded for storm {storm_name}...")
    return datasets

In [ ]:
# Function to limit data to plot area
def limit_data_to_plot_area(ds, lon_lat_limits):
    ds = ds.where((ds.longitude >= lon_lat_limits[0]) & (ds.longitude <= lon_lat_limits[1]) & 
                  (ds.latitude >= lon_lat_limits[2]) & (ds.latitude <= lon_lat_limits[3]), drop=True)
    return ds

# Function to define time steps
def get_time_steps(landfall):
    input_format = "%Y-%m-%d %H:%M:%S"
    landfall_time_index = datetime.strptime(landfall, input_format)
    return [landfall_time_index]

In [ ]:
def plot_data(datasets, storm_name, time_step):
    date_title = time_step.strftime("%Y-%m-%d %H")
    save_file_pdf = os.path.join(f"{storm_name}_storm_{date_title}.pdf")
    save_file_png = os.path.join(f"{storm_name}_storm_{date_title}.png")

    pplt.rc.reso = 'xx-hi'  # 'lo', 'med', 'hi', 'x-hi', 'xx-hi'
    proj1 = pplt.Proj('ortho', lon_0=storms[storm_name]["plot_lon_0"],
                      lat_0=storms[storm_name]["plot_lat_0"])  # "aea", "eqdc", "npstere", 'merc' 'tmerc' "robin"
    fig, axs = pplt.subplots(ncols=3, nrows=4, proj=proj1,
                             share=True, span=False, panelpad='1em')

    axs.format(lonlim=storms[storm_name]["plot_lon_lat_limits"][0:2],
               latlim=storms[storm_name]["plot_lon_lat_limits"][2:4],
               coast=True, coastlinewidth=0.5, lakes=False,
               rivers=False, labels=False, borders=True, labelsize=12,
               land=True, landcolor="lightgrey",
               linewidth=0.5, metacolor='black', alpha=0.5,
               #abc='a', abcloc='l', abcsize=15, abcweight='bold',
               toplabelweight="normal", toplabelsize=14,
               # suptitle=f"Storm {storm_name} at {date_title} UTC",
               suptitleweight="bold", suptitlesize=14,
               lonlocator=storms[storm_name]["lonlocator"],
               latlocator=storms[storm_name]["latlocator"],
               grid=True)
    
    labels = list("abcdefghijkl")  
    nrows, ncols = axs.shape  

    label_index = 0
    for col in range(ncols):  
        for row in range(nrows):  
            axs[row, col].text(0.04, 1.052, labels[label_index], transform=axs[row, col].transAxes, # 0.04, 1.052, For fiona
                            fontsize=15, fontweight='bold', va='center', ha='right')  
            label_index += 1

    # Calculate wind speed for the ds_uv10_wind and ds_uv10n_wind for time_index step
    datasets["uv10_wind"]["wind_speed"] = np.sqrt(
        datasets["uv10_wind"].sel(time=time_step)["u10"]**2 + datasets["uv10_wind"].sel(time=time_step)["v10"]**2)
    datasets["uv10n_wind"]["wind_speed"] = np.sqrt(
        datasets["uv10n_wind"].sel(time=time_step)["u10"]**2 + datasets["uv10n_wind"].sel(time=time_step)["v10"]**2)

    # Plotting settings
    wind_interval = storms[storm_name]["wind_interval"]
    wind_scale = storms[storm_name]["wind_scale"]
    wind_width = storms[storm_name]["wind_width"]
    levels_wind = storms[storm_name]["levels_wind"]
    
    u10_shade = axs[0, 0].contourf(datasets["uv10_wind"]["wind_speed"], cmap="magma", extend="max", levels=levels_wind)
    axs[0, 0].quiver(datasets["uv10_wind"]["longitude"][::wind_interval], datasets["uv10_wind"]["latitude"][::wind_interval],
                        datasets["uv10_wind"].sel(time=time_step)["u10"].values[::wind_interval, ::wind_interval],
                        datasets["uv10_wind"].sel(time=time_step)["v10"].values[::wind_interval, ::wind_interval],
                        scale=wind_scale, width=wind_width, color="black", alpha=0.8, pivot="tip", zorder=2)
    msl_contours = axs[0, 0].contour(datasets["msl"].sel(time=time_step)["msl"], colors="black", linewidths=0.8, levels=np.arange(920, 1060, 4))
    axs[0, 0].clabel(msl_contours, inline=True, fontsize=8, fmt="%1.0f")
    
    # Define the levels for the air density plot
    levels_rho = storms[storm_name]["levels_rho"]
    rho_shade = axs[1, 0].contourf(datasets["rho"].sel(time=time_step)["rho"], cmap="Spectral",
                                    extend="both", levels=levels_rho)
    
    # Difference of wind speeds from uv10n and uv10 (percentage?)
    wind_diff = datasets["uv10n_wind"]["wind_speed"] - datasets["uv10_wind"]["wind_speed"]
    wind_diff_shade = axs[2, 0].contourf(wind_diff, levels=np.arange(-0.8, 0.9, 0.1), cmap="div", extend="both")
    
    # Charnock shade vcenter=0.041*1000 or vcenter=0.018*1000 (from ecmwf)
    charnock_shade = axs[3, 0].contourf(datasets["charnock"].sel(time=time_step)["c"], cmap="viridis", extend="max",
                                        levels=storms[storm_name]["levels_charnock"])

    # Tau shades and differences for 2nd row
    tau_interval = storms[storm_name]["tau_interval"]
    tau_scale = storms[storm_name]["tau_scale"]
    tau_width = storms[storm_name]["tau_width"]
    levels_tau = storms[storm_name]["levels_tau"]

    rho_tau_shade = axs[0, 1].contourf(datasets["uv10rho_cd_tau"].sel(time=time_step)["tau"], levels=levels_tau,
                                        cmap="fire", extend="max")
    axs[0, 1].quiver(datasets["uv10rho_taux_tauy"]["longitude"][::tau_interval], datasets["uv10rho_taux_tauy"]["latitude"][::tau_interval],
                        datasets["uv10rho_taux_tauy"].sel(time=time_step)["taux"].values[::tau_interval, ::tau_interval],
                        datasets["uv10rho_taux_tauy"].sel(time=time_step)["tauy"].values[::tau_interval, ::tau_interval],
                        scale=tau_scale, width=tau_width, color="black", alpha=0.8, pivot="tip", zorder=2)
    
    uv10n_uv10rho_tau_diff = datasets["uv10n_cd_tau"].sel(time=time_step)["tau"] - datasets["uv10rho_cd_tau"].sel(time=time_step)["tau"]
    uv10n_uv10rho_tau_shade = axs[1, 1].contourf(uv10n_uv10rho_tau_diff, cmap="div", extend="both", levels=storms[storm_name]["tau_diff2"])
    axs[1, 1].quiver(datasets["uv10n_taux_tauy"]["longitude"][::tau_interval], datasets["uv10n_taux_tauy"]["latitude"][::tau_interval],
                        datasets["uv10n_taux_tauy"].sel(time=time_step)["taux"].values[::tau_interval, ::tau_interval],
                        datasets["uv10n_taux_tauy"].sel(time=time_step)["tauy"].values[::tau_interval, ::tau_interval],
                        scale=tau_scale, width=tau_width, color="black", alpha=0.8, pivot="tip", zorder=2)
    
    cv_uv10n_tau_diff = datasets["cv_cd_tau"].sel(time=time_step)["tau"] - datasets["uv10n_cd_tau"].sel(time=time_step)["tau"]
    cv_uv10n_tau_shade = axs[2, 1].contourf(cv_uv10n_tau_diff, cmap="div", extend="both", levels=storms[storm_name]["tau_diff2"])
    axs[2, 1].quiver(datasets["cv_taux_tauy"]["longitude"][::tau_interval], datasets["cv_taux_tauy"]["latitude"][::tau_interval],
                        datasets["cv_taux_tauy"].sel(time=time_step)["taux"].values[::tau_interval, ::tau_interval],
                        datasets["cv_taux_tauy"].sel(time=time_step)["tauy"].values[::tau_interval, ::tau_interval],
                        scale=tau_scale, width=tau_width, color="black", alpha=0.8, pivot="tip", zorder=2)

    cc_cv_tau_diff = datasets["cc_cd_tau"].sel(time=time_step)["tau"] - datasets["cv_cd_tau"].sel(time=time_step)["tau"]
    cc_cv_tau_shade = axs[3, 1].contourf(cc_cv_tau_diff, cmap="div", extend="both", levels=storms[storm_name]["tau_diff1"])
    axs[3, 1].quiver(datasets["cc_taux_tauy"]["longitude"][::tau_interval], datasets["cc_taux_tauy"]["latitude"][::tau_interval],
                        datasets["cc_taux_tauy"].sel(time=time_step)["taux"].values[::tau_interval, ::tau_interval],
                        datasets["cc_taux_tauy"].sel(time=time_step)["tauy"].values[::tau_interval, ::tau_interval],
                        scale=tau_scale, width=tau_width, color="black", alpha=0.8, pivot="tip", zorder=2)

    # Surge maps and normalized differences for 3rd row
    levels_surge = storms[storm_name]["levels_surge"]
    surge_dif = np.arange(-0.3, 0.31, 0.05)
    uv10rho_surge_shade = axs[0, 2].contourf(datasets["uv10rho_map"].sel(time=time_step)["surge"], levels=levels_surge,
                                        cmap="jet", extend="max")
    uv10n_uv10rho_surge_diff = datasets["uv10n_map"].sel(time=time_step)["surge"] - datasets["uv10rho_map"].sel(time=time_step)["surge"]
    uv10n_uv10rho_surge_shade = axs[1, 2].contourf(uv10n_uv10rho_surge_diff, cmap="div", extend="both", levels=surge_dif)
    cv_uv10n_surge_diff = datasets["cv_map"].sel(time=time_step)["surge"] - datasets["uv10n_map"].sel(time=time_step)["surge"]
    cv_uv10n_surge_shade = axs[2, 2].contourf(cv_uv10n_surge_diff, cmap="div", extend="both", levels=surge_dif)
    cc_cv_surge_diff = datasets["cc_map"].sel(time=time_step)["surge"] - datasets["cv_map"].sel(time=time_step)["surge"]
    cc_cv_surge_shade = axs[3, 2].contourf(cc_cv_surge_diff, cmap="div", extend="both", levels=surge_dif)

    ticks_wind_diff = np.arange(-0.8, 0.9, 0.2)
    ticks_surge_diff = np.arange(-0.3, 0.31, 0.1)

    # Create colorbars
    tls=11
    lbsize=11
    cb1 = fig.colorbar(u10_shade, label="Wind speed (m/s)", ax=axs[0, 0], ticklabelsize=tls, labelsize=lbsize,
                       pad=0.01, aspect=20)
    cb2 = fig.colorbar(rho_shade, label="Air density (kg/m³)", ax=axs[1, 0],
                    ticks=storms[storm_name]["ticks_rho"], ticklabelsize=tls, labelsize=lbsize,
                    pad=0.01, aspect=20)
    cb3 = fig.colorbar(wind_diff_shade, label="Wind speed difference (m/s)", ax=axs[2, 0],
                        ticks=ticks_wind_diff, ticklabelsize=tls, labelsize=lbsize,
                        pad=0.01, aspect=20)
    cb4 = fig.colorbar(charnock_shade, label="Charnock ($×10^{-3}$)", ax=axs[3, 0], ticklabelsize=tls, labelsize=lbsize,
                       pad=0.01, aspect=20)
    cb5 = fig.colorbar(rho_tau_shade, label="Wind stress (N/m²)", ax=axs[0, 1],
                        ticks=storms[storm_name]["ticks_tau"], ticklabelsize=tls, labelsize=lbsize,
                        pad=0.01, aspect=20)
    cb6 = fig.colorbar(uv10n_uv10rho_tau_shade, label="Wind stress difference (N/m²)", ax=axs[1, 1],
                        ticks=storms[storm_name]["ticks_tau_diff2"], ticklabelsize=tls, labelsize=lbsize,
                        pad=0.01, aspect=20)
    cb7 = fig.colorbar(cv_uv10n_tau_shade, label="Wind stress difference (N/m²)", ax=axs[2, 1],
                        ticks=storms[storm_name]["ticks_tau_diff2"], ticklabelsize=tls, labelsize=lbsize,
                        pad=0.01, aspect=20)
    cb8 = fig.colorbar(cc_cv_tau_shade, label="Wind stress difference (N/m²)", ax=axs[3, 1],
                        ticks=storms[storm_name]["ticks_tau_diff1"], ticklabelsize=tls, labelsize=lbsize,
                        pad=0.01, aspect=20)
    cb9 = fig.colorbar(uv10rho_surge_shade, label="Surge height (m)", ax=axs[0, 2],
                        ticks=storms[storm_name]["ticks_surge"], ticklabelsize=tls, labelsize=lbsize,
                        pad=0.01, aspect=20)
    cb10 = fig.colorbar(uv10n_uv10rho_surge_shade, label="Surge height difference (m)", ax=axs[1, 2],
                        ticks=ticks_surge_diff, ticklabelsize=tls, labelsize=lbsize,
                        pad=0.01, aspect=20)
    cb11 = fig.colorbar(cv_uv10n_surge_shade, label="Surge height difference (m)", ax=axs[2, 2],
                        ticks=ticks_surge_diff, ticklabelsize=tls, labelsize=lbsize,
                        pad=0.01, aspect=20)
    cb12 = fig.colorbar(cc_cv_surge_shade, label="Surge height difference (m)", ax=axs[3, 2],
                        ticks=ticks_surge_diff, ticklabelsize=tls, labelsize=lbsize,
                        pad=0.01, aspect=20)

    # Create ltitles
    ltsize = 13.5
    ltsize2 = 12.5
    axs[0, 0].format(ltitle="   Wind speed ($U_{10}$)", titlesize=ltsize)
    axs[1, 0].format(ltitle="   Air density ($ρ_{a}$)", titlesize=ltsize)
    axs[2, 0].format(ltitle="   Wind speed difference ($ΔU_{10}$)", titlesize=ltsize)
    axs[3, 0].format(ltitle="   Charnock parameter ($α_{Chvar}$)", titlesize=ltsize)
    axs[0, 1].format(ltitle="   Wind stress ($GTSM_{IFS}$)", titlesize=ltsize)
    axs[1, 1].format(ltitle="   ΔStress ($GTSM_{CAD}$—$GTSM_{IFS}$)", titlesize=ltsize2)
    axs[2, 1].format(ltitle="   ΔStress ($GTSM_{NST}$—$GTSM_{CAD}$)", titlesize=ltsize2)
    axs[3, 1].format(ltitle="   ΔStress ($GTSM_{FIX}$—$GTSM_{NST}$)", titlesize=ltsize2)
    axs[0, 2].format(ltitle="   Surge height ($GTSM_{IFS}$)", titlesize=ltsize)
    axs[1, 2].format(ltitle="   ΔSurge ($GTSM_{CAD}$—$GTSM_{IFS}$)", titlesize=ltsize2)
    axs[2, 2].format(ltitle="   ΔSurge ($GTSM_{NST}$—$GTSM_{CAD}$)", titlesize=ltsize2)
    axs[3, 2].format(ltitle="   ΔSurge ($GTSM_{FIX}$—$GTSM_{NST}$)", titlesize=ltsize2)

    axs[:, 0].format(latlabels='l', latlocator=storms[storm_name]["latlocator"])
    axs[3, :].format(lonlabels="b", lonlocator=storms[storm_name]["lonlocator"])
    
    # Save the plot
    # Save the plot as PDF and PNG, cropping the bounding box
    # Define the target directory
    save_dir = 'final_plots'
    os.makedirs(save_dir, exist_ok=True)  # Create the directory if it doesn't exist

    # Save the plot as PDF and PNG in the specified directory
    fig.savefig(os.path.join(save_dir, f'Storm_{storm_name}_panel_vars_4.pdf'), bbox_inches='tight')
    fig.savefig(os.path.join(save_dir, f'Storm_{storm_name}_panel_vars_4.png'), dpi=600, bbox_inches='tight')

    # Show the horizontal plot
    plt.show()

In [ ]:
# Plot Xaver storm
storm_name = "Xaver"
nc_read = storms[storm_name]["nc_read"]
datasets = load_data(storm_name, nc_read)
time_steps = get_time_steps(storms[storm_name]["landfall"])
time_step = time_steps[0]
datasets = {key: limit_data_to_plot_area(datasets[key], storms[storm_name]["filter_lon_lat_limits"]) for key in datasets.keys()}
plot_data(datasets, storm_name, time_step)

In [ ]:
# Plot for fiona storm
storm_name = "Fiona"
nc_read = storms[storm_name]["nc_read"]
datasets = load_data(storm_name, nc_read)
time_steps = get_time_steps(storms[storm_name]["landfall"])
time_step = time_steps[0]
datasets = {key: limit_data_to_plot_area(datasets[key], storms[storm_name]["filter_lon_lat_limits"]) for key in datasets.keys()}
plot_data(datasets, storm_name, time_step)